## 2.1 卷积层计算
输出特征图尺寸计算：
高度和宽度的计算公式为：
  输出高度 = (输入高度 + 2 × 填充 - 卷积核大小) / 步幅 + 1
  输出宽度 = (输入宽度 + 2 × 填充 - 卷积核大小) / 步幅 + 1

代入数据：输入高度和宽度均为 32，卷积核大小 5，填充 2，步幅 2
  输出高度 = (32 + 2 × 2 - 5) / 2 + 1 = 16
  输出宽度 = (32 + 2 × 2 - 5) / 2 + 1 = 16

通道数等于卷积核数量 = 16
输出特征图尺寸为 16 × 16 × 16

单个输出通道一个像素点乘次数：
 卷积核大小 = 3 × 5 × 5 = 75
 每个输出像素需要对输入局部区域做 75 次乘法操作

每个输出像素需要 75 次乘法**


## 3.1 VGG 卷积参数量计算
1. 一个 5 × 5 卷积层参数量（不带偏置）：
* 参数量 = 输入通道数 × 输出通道数 × 卷积核大小 × 卷积核大小
* 代入数值 = C × C × 5 × 5 = 25 × C²

2. 两个串联的 3 × 3 卷积层参数量（不带偏置，通道数均为 C）：
* 每层参数量 = C × C × 3 × 3 = 9 × C²
* 两层总参数量 = 9 × C² + 9 × C² = 18 × C²

**两个 3 × 3 卷积层的参数量比一个 5 × 5 卷积层少，并且感受野相同**


## 4.1 Batch Normalization 计算
1. 计算均值：

* 均值 = (2 + 4 + 6 + 8) / 4 = 5

2. 计算方差：

* 方差 = ((2-5)² + (4-5)² + (6-5)² + (8-5)²) / 4
* = (9 + 1 + 1 + 9) / 4 = 20 / 4 = 5

3. 标准化每个样本：

* 样本标准化 = (x - 均值) / √方差 = (x - 5) / √5

4. 缩放和平移：

* 输出 y = γ × 标准化 + β = 2 × (x - 5)/√5 + 1

5. 计算结果：

* x1 = 2 → y1 ≈ -1.683
* x2 = 4 → y2 ≈ 0.105
* x3 = 6 → y3 ≈ 1.895
* x4 = 8 → y4 ≈ 3.683


## 5.1 微调理论
1. 为什么底层特征层学习率小：
* 底层卷积层提取的是通用特征（边缘、纹理、颜色模式等），对大多数任务都有效
* 如果学习率太大，会破坏预训练好的通用特征
* 顶层输出层新初始化，需要较大学习率快速收敛

2. 小数据集且与源数据集相似时的微调策略：
* 冻结底层卷积层，只训练顶层输出层
* 或者给底层使用较小的学习率微调
* 结合数据增强以防止过拟合

## 6.1 目标检测 IoU 计算
1. 交集面积：
* 左上角坐标 = (max(10,30), max(10,30)) = (30,30)
* 右下角坐标 = (min(50,70), min(50,70)) = (50,50)
* 宽度 = 50 - 30 = 20
* 高度 = 50 - 30 = 20
* 面积 = 20 × 20 = 400

2. 并集面积：
* A 面积 = (50-10) × (50-10) = 40 × 40 = 1600
* B 面积 = 40 × 40 = 1600
* 并集面积 = 1600 + 1600 - 400 = 2800

3. IoU = 交集面积 / 并集面积 = 400 / 2800 ≈ 0.1429

**IoU ≈ 0.1429**

In [1]:
import numpy as np

def max_pool2d(x, kernel_size, stride=1, padding=0):
    """
    x: 输入特征图，形状(H, W)
    kernel_size: 池化窗口大小
    stride: 步幅
    padding: 填充大小
    """

    # Padding
    x_padded = np.pad(
        x,
        ((padding, padding), (padding, padding)),
        mode='constant',
        constant_values=0
    )

    H, W = x_padded.shape

    out_h = (H - kernel_size) // stride + 1
    out_w = (W - kernel_size) // stride + 1

    output = np.zeros((out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            h_start = i * stride
            h_end = h_start + kernel_size

            w_start = j * stride
            w_end = w_start + kernel_size

            window = x_padded[h_start:h_end, w_start:w_end]

            output[i, j] = np.max(window)

    return output


# 测试
x = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [7, 8, 3, 1],
    [4, 2, 5, 6]
])

result = max_pool2d(
    x,
    kernel_size=2,
    stride=2,
    padding=0
)

print(result)

[[6. 4.]
 [8. 6.]]


In [2]:
import torch
from torch import nn


def nin_block(in_channels,
              out_channels,
              kernel_size,
              stride,
              padding):

    return nn.Sequential(
        nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        ),
        nn.ReLU(),

        nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=1
        ),
        nn.ReLU(),

        nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=1
        ),
        nn.ReLU()
    )


# 测试
X = torch.randn(1, 3, 224, 224)

net = nin_block(
    in_channels=3,
    out_channels=96,
    kernel_size=11,
    stride=4,
    padding=0
)

Y = net(X)

print(Y.shape)

torch.Size([1, 96, 54, 54])


In [4]:
import torch
from torch import nn
import torch.nn.functional as F


class Residual(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        use_1x1conv=False,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            stride=stride
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        if use_1x1conv:
            self.conv3 = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride
            )
        else:
            self.conv3 = None

    def forward(self, X):

        Y = F.relu(
            self.bn1(
                self.conv1(X)
            )
        )

        Y = self.bn2(
            self.conv2(Y)
        )

        if self.conv3:
            X = self.conv3(X)

        Y += X

        return F.relu(Y)


# 测试
X = torch.randn(4, 3, 64, 64)

blk = Residual(
    in_channels=3,
    out_channels=16,
    use_1x1conv=True,
    stride=2
)

Y = blk(X)

print(Y.shape)

torch.Size([4, 16, 32, 32])


In [5]:
from torchvision import transforms


transform = transforms.Compose([

    # 随机裁剪
    transforms.RandomResizedCrop(
        size=224,
        scale=(0.08, 1.0)
    ),

    # 50%概率水平翻转
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    # 亮度、对比度、饱和度变化
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.5
    ),

    # 转Tensor
    transforms.ToTensor()
])

print(transform)

Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


In [6]:
import torch
import torch.nn.functional as F


def label_smoothing_cross_entropy(
    logits,
    target,
    epsilon=0.1
):
    """
    logits: (batch_size, num_classes)
    target: (batch_size,)
    """

    num_classes = logits.size(1)

    log_probs = F.log_softmax(
        logits,
        dim=1
    )

    smooth_labels = torch.full_like(
        log_probs,
        epsilon / (num_classes - 1)
    )

    smooth_labels.scatter_(
        1,
        target.unsqueeze(1),
        1 - epsilon
    )

    loss = -(smooth_labels * log_probs).sum(dim=1)

    return loss.mean()


# 测试
logits = torch.tensor([
    [2.0, 0.5, 0.3],
    [0.2, 1.5, 0.1]
])

target = torch.tensor([0, 1])

loss = label_smoothing_cross_entropy(
    logits,
    target,
    epsilon=0.1
)

print("Loss =", loss.item())

Loss = 0.5268766283988953
